# 03 — Bitcoin Spot and Deribit DVOL Data Collection

This notebook constructs the Bitcoin spot-price and Deribit DVOL inputs used to estimate derivatives-implied benchmark probabilities.

The main daily dataset combines Bitcoin spot prices with Deribit's DVOL index over the thesis sample period. Daily observations are used for the Polymarket comparison under a conservative availability convention that avoids look-ahead.

The notebook also constructs a timestamp-aligned benchmark for the supplementary Kalshi analysis. For each Kalshi first-trade observation, Bitcoin spot is measured using the latest completed one-minute Binance candle, while volatility is measured using the latest daily DVOL close available before the trade.

The notebook does not reconstruct a full historical Deribit option chain or estimate strike-specific implied volatilities.

.

In [36]:
# ============================================================
# Imports and project configuration
# ============================================================

import os
import time
import requests

import numpy as np
import pandas as pd

from importlib import reload
import config
reload(config)

from config import *

print("DOWNLOAD_DERIBIT:", DOWNLOAD_DERIBIT)
print(
    "DOWNLOAD_KALSHI_INTRADAY_SPOT:",
    DOWNLOAD_KALSHI_INTRADAY_SPOT,
)
print("SAMPLE_START:", SAMPLE_START)
print("SAMPLE_END:", SAMPLE_END)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("FINAL_DIR:", FINAL_DIR)
for folder in [RAW_DIR, PROCESSED_DIR, FINAL_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DOWNLOAD_DERIBIT: False
DOWNLOAD_KALSHI_INTRADAY_SPOT: False
SAMPLE_START: 2024-01-01
SAMPLE_END: 2026-06-04
RAW_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw
PROCESSED_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed
FINAL_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final


In [9]:
# ============================================================
# API settings
# ============================================================

BINANCE_KLINES_URL = "https://api.binance.com/api/v3/klines"

DERIBIT_BASE_URL = "https://www.deribit.com/api/v2"
DERIBIT_DVOL_URL = f"{DERIBIT_BASE_URL}/public/get_volatility_index_data"

REQUEST_DELAY = 0.20

In [13]:
# ============================================================
# Download BTC spot daily from Binance
# ============================================================

btc_spot_raw_path = RAW_DIR / "btc_spot_daily.csv"

def fetch_btc_spot_binance(from_date, to_date):
    start_ms = int(pd.Timestamp(from_date, tz="UTC").timestamp() * 1000)
    end_ms = int((pd.Timestamp(to_date, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000)

    rows = []
    current_ms = start_ms

    while current_ms < end_ms:
        response = requests.get(
            BINANCE_KLINES_URL,
            params={
                "symbol": "BTCUSDT",
                "interval": "1d",
                "startTime": current_ms,
                "endTime": end_ms,
                "limit": 1000,
            },
            timeout=30,
        )

        response.raise_for_status()
        data = response.json()

        if not data:
            break

        chunk = pd.DataFrame(
            data,
            columns=[
                "open_ts", "open", "high", "low", "close", "volume",
                "close_ts", "quote_volume", "n_trades",
                "taker_buy_base", "taker_buy_quote", "ignore",
            ],
        )

        chunk["date"] = pd.to_datetime(chunk["open_ts"], unit="ms", utc=True).dt.normalize()
        chunk["spot"] = pd.to_numeric(chunk["close"], errors="coerce")

        rows.append(chunk[["date", "spot"]])

        print(
            f"Binance: +{len(chunk):,} rows "
            f"[{chunk['date'].iloc[0].date()} -> {chunk['date'].iloc[-1].date()}]"
        )

        current_ms = int(data[-1][0]) + 86_400_000

        if len(data) < 1000:
            break

        time.sleep(REQUEST_DELAY)

    if not rows:
        return pd.DataFrame(columns=["date", "spot"])

    return (
        pd.concat(rows, ignore_index=True)
        .drop_duplicates("date")
        .sort_values("date")
        .reset_index(drop=True)
    )


if DOWNLOAD_DERIBIT:
    print("Downloading BTC spot daily...")

    df_spot = fetch_btc_spot_binance(SAMPLE_START, SAMPLE_END)

    if df_spot.empty:
        raise ValueError("No BTC spot data downloaded.")

    df_spot.to_csv(btc_spot_raw_path, index=False)
    print(f"Saved BTC spot to: {btc_spot_raw_path}")

else:
    if not btc_spot_raw_path.exists():
        raise FileNotFoundError(
            f"BTC spot file not found: {btc_spot_raw_path}\n"
            "Set DOWNLOAD_DERIBIT = True in config.py to download it."
        )

    df_spot = pd.read_csv(btc_spot_raw_path)
    df_spot["date"] = pd.to_datetime(df_spot["date"], utc=True)
    print(f"Loaded BTC spot from: {btc_spot_raw_path}")

print("=" * 70)
print("BTC spot daily")
print("=" * 70)
print(f"Rows: {len(df_spot):,}")
print("Date range:", df_spot["date"].min(), "->", df_spot["date"].max())
print("Missing spot:", df_spot["spot"].isna().sum())
print("Spot min/max:", df_spot["spot"].min(), df_spot["spot"].max())

Loaded BTC spot from: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/btc_spot_daily.csv
BTC spot daily
Rows: 887
Date range: 2024-01-01 00:00:00+00:00 -> 2026-06-05 00:00:00+00:00
Missing spot: 0
Spot min/max: 39568.02 124658.54


In [15]:
# ============================================================
# Download Deribit BTC DVOL daily
# ============================================================

btc_dvol_raw_path = RAW_DIR / "btc_dvol_daily.csv"

def fetch_deribit_dvol_daily(from_date, to_date):
    start_ms = int(pd.Timestamp(from_date, tz="UTC").timestamp() * 1000)
    end_ms = int((pd.Timestamp(to_date, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000)

    response = requests.get(
        DERIBIT_DVOL_URL,
        params={
            "currency": "BTC",
            "start_timestamp": start_ms,
            "end_timestamp": end_ms,
            "resolution": "1D",
        },
        timeout=30,
    )

    response.raise_for_status()
    payload = response.json()

    if "error" in payload:
        raise RuntimeError(f"Deribit API error: {payload['error']}")

    result = payload.get("result", {})
    data = result.get("data", [])

    if not data:
        return pd.DataFrame(columns=["date", "dvol_open", "dvol_high", "dvol_low", "dvol"])

    df = pd.DataFrame(
        data,
        columns=["timestamp", "dvol_open", "dvol_high", "dvol_low", "dvol"],
    )

    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.normalize()

    for col in ["dvol_open", "dvol_high", "dvol_low", "dvol"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return (
        df[["date", "dvol_open", "dvol_high", "dvol_low", "dvol"]]
        .drop_duplicates("date")
        .sort_values("date")
        .reset_index(drop=True)
    )


if DOWNLOAD_DERIBIT:
    print("Downloading Deribit BTC DVOL daily...")

    df_dvol = fetch_deribit_dvol_daily(SAMPLE_START, SAMPLE_END)

    if df_dvol.empty:
        raise ValueError("No Deribit DVOL data downloaded.")

    df_dvol.to_csv(btc_dvol_raw_path, index=False)
    print(f"Saved BTC DVOL to: {btc_dvol_raw_path}")

else:
    if not btc_dvol_raw_path.exists():
        raise FileNotFoundError(
            f"BTC DVOL file not found: {btc_dvol_raw_path}\n"
            "Set DOWNLOAD_DERIBIT = True in config.py to download it."
        )

    df_dvol = pd.read_csv(btc_dvol_raw_path)
    df_dvol["date"] = pd.to_datetime(df_dvol["date"], utc=True)
    print(f"Loaded BTC DVOL from: {btc_dvol_raw_path}")

print("=" * 70)
print("Deribit BTC DVOL daily")
print("=" * 70)
print(f"Rows: {len(df_dvol):,}")
print("Date range:", df_dvol["date"].min(), "->", df_dvol["date"].max())
print("Missing DVOL:", df_dvol["dvol"].isna().sum())
print("DVOL min/max:", df_dvol["dvol"].min(), df_dvol["dvol"].max())

Loaded BTC DVOL from: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/btc_dvol_daily.csv
Deribit BTC DVOL daily
Rows: 887
Date range: 2024-01-01 00:00:00+00:00 -> 2026-06-05 00:00:00+00:00
Missing DVOL: 0
DVOL min/max: 33.81 83.02


In [17]:
# ============================================================
# Merge BTC spot and Deribit DVOL daily
# ============================================================

df_spot["date"] = pd.to_datetime(df_spot["date"], utc=True).dt.normalize()
df_dvol["date"] = pd.to_datetime(df_dvol["date"], utc=True).dt.normalize()

sample_start_dt = pd.Timestamp(SAMPLE_START, tz="UTC")
sample_end_dt = pd.Timestamp(SAMPLE_END, tz="UTC")

df_deribit_daily = (
    df_spot
    .merge(df_dvol, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)

df_deribit_daily = df_deribit_daily[
    (df_deribit_daily["date"] >= sample_start_dt) &
    (df_deribit_daily["date"] <= sample_end_dt)
].copy()

df_deribit_daily["sigma"] = df_deribit_daily["dvol"] / 100

print("=" * 70)
print("Deribit daily inputs")
print("=" * 70)

print(f"Rows: {len(df_deribit_daily):,}")
print("Date range:", df_deribit_daily["date"].min(), "->", df_deribit_daily["date"].max())

print("\nMissing values:")
print(df_deribit_daily[["spot", "dvol", "sigma"]].isna().sum())

print("\nSpot summary:")
display(df_deribit_daily["spot"].describe())

print("\nDVOL summary:")
display(df_deribit_daily[["dvol", "sigma"]].describe())

display(df_deribit_daily.head())
display(df_deribit_daily.tail())

Deribit daily inputs
Rows: 886
Date range: 2024-01-01 00:00:00+00:00 -> 2026-06-04 00:00:00+00:00

Missing values:
spot     0
dvol     0
sigma    0
dtype: int64

Spot summary:


count       886.000000
mean      82422.673623
std       20732.772296
min       39568.020000
25%       66076.527500
50%       81102.885000
75%       99295.200000
max      124658.540000
Name: spot, dtype: float64


DVOL summary:


,dvol,sigma
count,886.000000,886.000000
mean,50.890564,0.508906
std,9.710996,0.097110
min,33.810000,0.338100
25%,43.287500,0.432875
50%,51.040000,0.510400
75%,56.790000,0.567900
max,83.020000,0.830200


,date,spot,dvol_open,dvol_high,dvol_low,dvol,sigma
0,2024-01-01 00:00:00+00:00,44179.55,64.71,66.89,63.88,66.81,0.6681
1,2024-01-02 00:00:00+00:00,44946.91,66.81,68.04,63.73,63.75,0.6375
2,2024-01-03 00:00:00+00:00,42845.23,63.75,67.12,62.13,65.17,0.6517
3,2024-01-04 00:00:00+00:00,44151.10,65.17,66.59,63.74,65.34,0.6534
4,2024-01-05 00:00:00+00:00,44145.11,65.34,68.25,65.27,67.64,0.6764


,date,spot,dvol_open,dvol_high,dvol_low,dvol,sigma
881,2026-05-31 00:00:00+00:00,73674.39,35.28,36.53,35.19,36.40,0.3640
882,2026-06-01 00:00:00+00:00,71408.90,36.40,38.22,36.05,37.27,0.3727
883,2026-06-02 00:00:00+00:00,66760.83,37.27,43.36,37.27,43.28,0.4328
884,2026-06-03 00:00:00+00:00,64142.75,43.28,47.90,42.29,47.90,0.4790
885,2026-06-04 00:00:00+00:00,63885.99,47.90,53.28,46.16,46.17,0.4617


In [19]:
# ============================================================
# Deribit daily input diagnostics
# ============================================================

print("=" * 70)
print("Deribit daily input diagnostics")
print("=" * 70)

print("Dataset size:")
print(f"Rows: {len(df_deribit_daily):,}")
print(f"Unique dates: {df_deribit_daily['date'].nunique():,}")

print("\nDate range:")
print("Min date:", df_deribit_daily["date"].min())
print("Max date:", df_deribit_daily["date"].max())

print("\nMissing values:")
print(df_deribit_daily[["date", "spot", "dvol", "sigma"]].isna().sum())

print("\nDuplicate dates:")
print(df_deribit_daily.duplicated(subset=["date"]).sum())

print("\nInvalid values:")
print("Spot <= 0:", (df_deribit_daily["spot"] <= 0).sum())
print("DVOL <= 0:", (df_deribit_daily["dvol"] <= 0).sum())
print("Sigma <= 0:", (df_deribit_daily["sigma"] <= 0).sum())

print("\nYearly summary:")
display(
    df_deribit_daily.assign(year=df_deribit_daily["date"].dt.year)
    .groupby("year")
    .agg(
        n_days=("date", "nunique"),
        avg_spot=("spot", "mean"),
        min_spot=("spot", "min"),
        max_spot=("spot", "max"),
        avg_dvol=("dvol", "mean"),
        min_dvol=("dvol", "min"),
        max_dvol=("dvol", "max"),
    )
    .round(3)
)

Deribit daily input diagnostics
Dataset size:
Rows: 886
Unique dates: 886

Date range:
Min date: 2024-01-01 00:00:00+00:00
Max date: 2026-06-04 00:00:00+00:00

Missing values:
date     0
spot     0
dvol     0
sigma    0
dtype: int64

Duplicate dates:
0

Invalid values:
Spot <= 0: 0
DVOL <= 0: 0
Sigma <= 0: 0

Yearly summary:


,n_days,avg_spot,min_spot,max_spot,avg_dvol,min_dvol,max_dvol
year,,,,,,,
2024,366,65963.593,39568.02,106133.74,57.710,41.52,83.02
2025,365,101635.673,76322.42,124658.54,46.035,33.81,66.02
2026,155,76043.828,62909.86,96951.78,46.221,34.44,82.62


In [21]:
# ============================================================
# Save Deribit daily input datasets
# ============================================================

deribit_daily_processed_path = PROCESSED_DIR / "deribit_btc_daily_inputs.csv"
deribit_daily_final_path = FINAL_DIR / "deribit_btc_daily_inputs.csv"
deribit_summary_path = TABLES_DIR / "deribit_daily_input_summary.csv"

df_deribit_daily.to_csv(deribit_daily_processed_path, index=False)
df_deribit_daily.to_csv(deribit_daily_final_path, index=False)

deribit_summary = pd.DataFrame({
    "dataset": ["Deribit daily inputs"],
    "rows": [len(df_deribit_daily)],
    "start_date": [df_deribit_daily["date"].min()],
    "end_date": [df_deribit_daily["date"].max()],
    "missing_spot": [df_deribit_daily["spot"].isna().sum()],
    "missing_dvol": [df_deribit_daily["dvol"].isna().sum()],
    "avg_spot": [df_deribit_daily["spot"].mean()],
    "avg_dvol": [df_deribit_daily["dvol"].mean()],
    "avg_sigma": [df_deribit_daily["sigma"].mean()],
})

deribit_summary.to_csv(deribit_summary_path, index=False)

print("=" * 70)
print("Deribit daily inputs saved")
print("=" * 70)

print(f"Processed dataset: {deribit_daily_processed_path}")
print(f"Final dataset: {deribit_daily_final_path}")
print(f"Summary table: {deribit_summary_path}")

display(deribit_summary)

Deribit daily inputs saved
Processed dataset: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/deribit_btc_daily_inputs.csv
Final dataset: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/deribit_btc_daily_inputs.csv
Summary table: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/deribit_daily_input_summary.csv


,dataset,rows,start_date,end_date,missing_spot,missing_dvol,avg_spot,avg_dvol,avg_sigma
0,Deribit daily inputs,886,2024-01-01 00:00:00+00:00,2026-06-04 00:00:00+00:00,0,0,82422.673623,50.890564,0.508906


In [23]:
# ============================================================
# Load timestamped Kalshi observations for intraday benchmark
# ============================================================

import requests
import time

kalshi_timestamped_path = (
    FINAL_DIR / "kalshi_kxbtc_first_trades_timestamped_30m.csv"
)

df_kalshi_targets = pd.read_csv(
    kalshi_timestamped_path,
    low_memory=False,
)

for col in ["observation_time", "open_time", "close_time"]:
    df_kalshi_targets[col] = pd.to_datetime(
        df_kalshi_targets[col],
        utc=True,
        errors="coerce",
    )

assert len(df_kalshi_targets) == 3_135
assert df_kalshi_targets["ticker"].is_unique
assert df_kalshi_targets["observation_time"].notna().all()
assert df_kalshi_targets["event_ticker"].notna().all()

kalshi_event_windows = (
    df_kalshi_targets
    .groupby("event_ticker", as_index=False)
    .agg(
        window_start=("open_time", "min"),
        window_end=("observation_time", "max"),
        observations=("ticker", "size"),
    )
)

# Include several completed minutes before market opening.
kalshi_event_windows["window_start"] = (
    kalshi_event_windows["window_start"]
    .dt.floor("min")
    - pd.Timedelta(minutes=5)
)

kalshi_event_windows["window_end"] = (
    kalshi_event_windows["window_end"]
    .dt.ceil("min")
)

print("=" * 70)
print("Kalshi intraday benchmark targets")
print("=" * 70)
print(f"Observations: {len(df_kalshi_targets):,}")
print(f"Events: {len(kalshi_event_windows):,}")
print(
    "Observation range:",
    df_kalshi_targets["observation_time"].min(),
    "->",
    df_kalshi_targets["observation_time"].max(),
)

display(kalshi_event_windows.head())

Kalshi intraday benchmark targets
Observations: 3,135
Events: 485
Observation range: 2025-02-04 04:00:28.774289+00:00 -> 2026-06-04 03:27:16.183834+00:00


,event_ticker,window_start,window_end,observations
0,KXBTC-25APR0100,2025-04-01 02:55:00+00:00,2025-04-01 03:19:00+00:00,4
1,KXBTC-25APR0200,2025-04-02 02:55:00+00:00,2025-04-02 03:23:00+00:00,3
2,KXBTC-25APR0300,2025-04-03 02:55:00+00:00,2025-04-03 03:24:00+00:00,7
3,KXBTC-25APR0400,2025-04-04 02:55:00+00:00,2025-04-04 03:23:00+00:00,2
4,KXBTC-25APR0500,2025-04-05 02:55:00+00:00,2025-04-05 03:28:00+00:00,4


In [25]:
# ============================================================
# Download Binance one-minute spot windows for Kalshi
# ============================================================

BINANCE_KLINES_URL = "https://api.binance.com/api/v3/klines"
BINANCE_SYMBOL = "BTCUSDT"
BINANCE_INTERVAL = "1m"

kalshi_spot_1m_raw_path = (
    RAW_DIR / "binance_btcusdt_1m_kalshi_windows.csv"
)

binance_kline_columns = [
    "open_time_ms",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time_ms",
    "quote_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore",
]


def fetch_binance_one_minute_window(
    event_ticker,
    start_time,
    end_time,
):
    response = requests.get(
        BINANCE_KLINES_URL,
        params={
            "symbol": BINANCE_SYMBOL,
            "interval": BINANCE_INTERVAL,
            "startTime": int(start_time.timestamp() * 1000),
            "endTime": int(end_time.timestamp() * 1000),
            "limit": 1000,
        },
        timeout=30,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Binance request failed for {event_ticker}. "
            f"HTTP {response.status_code}: "
            f"{response.text[:500]}"
        )

    data = response.json()

    if not isinstance(data, list):
        raise RuntimeError(
            f"Unexpected Binance response for {event_ticker}: "
            f"{str(data)[:500]}"
        )

    if not data:
        return pd.DataFrame()

    frame = pd.DataFrame(
        data,
        columns=binance_kline_columns,
    )

    frame["event_ticker"] = event_ticker

    return frame


if DOWNLOAD_KALSHI_INTRADAY_SPOT:
    spot_frames = []

    print(
        f"Downloading one-minute BTC spot windows for "
        f"{len(kalshi_event_windows):,} Kalshi events..."
    )

    for i, row in enumerate(
        kalshi_event_windows.itertuples(index=False),
        start=1,
    ):
        frame = fetch_binance_one_minute_window(
            event_ticker=row.event_ticker,
            start_time=row.window_start,
            end_time=row.window_end,
        )

        if frame.empty:
            raise ValueError(
                f"No Binance candles returned for "
                f"{row.event_ticker}."
            )

        spot_frames.append(frame)

        if (
            i == 1
            or i % 25 == 0
            or i == len(kalshi_event_windows)
        ):
            print(
                f"Processed {i:>3}/"
                f"{len(kalshi_event_windows):>3} events"
            )

        time.sleep(0.10)

    df_kalshi_spot_1m = pd.concat(
        spot_frames,
        ignore_index=True,
    )

    df_kalshi_spot_1m.to_csv(
        kalshi_spot_1m_raw_path,
        index=False,
    )

    print("\nSaved raw one-minute spot to:")
    print(kalshi_spot_1m_raw_path)

else:
    if not kalshi_spot_1m_raw_path.exists():
        raise FileNotFoundError(
            "Kalshi intraday spot file not found:\n"
            f"{kalshi_spot_1m_raw_path}\n"
            "Set DOWNLOAD_KALSHI_INTRADAY_SPOT = True."
        )

    df_kalshi_spot_1m = pd.read_csv(
        kalshi_spot_1m_raw_path,
        low_memory=False,
    )

    print("Loaded one-minute spot from:")
    print(kalshi_spot_1m_raw_path)


# ------------------------------------------------------------
# Clean and validate one-minute candles
# ------------------------------------------------------------

df_kalshi_spot_1m["spot_open_time"] = pd.to_datetime(
    df_kalshi_spot_1m["open_time_ms"],
    unit="ms",
    utc=True,
    errors="coerce",
)

df_kalshi_spot_1m["spot_available_at"] = pd.to_datetime(
    df_kalshi_spot_1m["close_time_ms"],
    unit="ms",
    utc=True,
    errors="coerce",
)

df_kalshi_spot_1m["spot"] = pd.to_numeric(
    df_kalshi_spot_1m["close"],
    errors="coerce",
)

df_kalshi_spot_1m = (
    df_kalshi_spot_1m
    .sort_values(["event_ticker", "spot_available_at"])
    .drop_duplicates(
        subset=["event_ticker", "spot_available_at"],
        keep="last",
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("Binance one-minute Kalshi spot windows")
print("=" * 70)
print(f"Rows: {len(df_kalshi_spot_1m):,}")
print(
    f"Events: "
    f"{df_kalshi_spot_1m['event_ticker'].nunique():,}"
)
print(
    "Time range:",
    df_kalshi_spot_1m["spot_available_at"].min(),
    "->",
    df_kalshi_spot_1m["spot_available_at"].max(),
)
print("Missing spot:", df_kalshi_spot_1m["spot"].isna().sum())
print("Invalid spot:", (df_kalshi_spot_1m["spot"] <= 0).sum())

assert (
    df_kalshi_spot_1m["event_ticker"].nunique()
    == 485
)
assert df_kalshi_spot_1m["spot"].notna().all()
assert (df_kalshi_spot_1m["spot"] > 0).all()

Processed   1/485 events
Processed  25/485 events
Processed  50/485 events
Processed  75/485 events
Processed 100/485 events
Processed 125/485 events
Processed 150/485 events
Processed 175/485 events
Processed 200/485 events
Processed 225/485 events
Processed 250/485 events
Processed 275/485 events
Processed 300/485 events
Processed 325/485 events
Processed 350/485 events
Processed 375/485 events
Processed 400/485 events
Processed 425/485 events
Processed 450/485 events
Processed 475/485 events
Processed 485/485 events

Saved raw one-minute spot to:
/Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/binance_btcusdt_1m_kalshi_windows.csv
Binance one-minute Kalshi spot windows
Rows: 11,351
Events: 485
Time range: 2025-02-04 03:55:59.999000+00:00 -> 2026-06-04 03:28:59.999000+00:00
Missing spot: 0
Invalid spot: 0


In [26]:
# ============================================================
# Construct temporally available Kalshi intraday benchmark
# ============================================================

# ------------------------------------------------------------
# Match latest completed one-minute candle within each event
# ------------------------------------------------------------

spot_matches = []

for event_ticker, event_targets in (
    df_kalshi_targets.groupby("event_ticker")
):
    event_spot = df_kalshi_spot_1m.loc[
        df_kalshi_spot_1m["event_ticker"]
        == event_ticker,
        ["spot_available_at", "spot"],
    ].copy()

    event_targets = event_targets.sort_values(
        "observation_time"
    )

    event_spot = event_spot.sort_values(
        "spot_available_at"
    )

    event_matched = pd.merge_asof(
        event_targets,
        event_spot,
        left_on="observation_time",
        right_on="spot_available_at",
        direction="backward",
        tolerance=pd.Timedelta(minutes=5),
    )

    spot_matches.append(event_matched)

df_kalshi_intraday = pd.concat(
    spot_matches,
    ignore_index=True,
)

df_kalshi_intraday["spot_age_seconds"] = (
    (
        df_kalshi_intraday["observation_time"]
        - df_kalshi_intraday["spot_available_at"]
    ).dt.total_seconds()
)


# ------------------------------------------------------------
# Build latest available daily DVOL input
# ------------------------------------------------------------

df_dvol_available = (
    df_deribit_daily[
        ["date", "dvol", "sigma"]
    ]
    .copy()
)

df_dvol_available["benchmark_source_date"] = (
    pd.to_datetime(
        df_dvol_available["date"],
        utc=True,
        errors="coerce",
    )
    .dt.normalize()
)

df_dvol_available["benchmark_available_at"] = (
    df_dvol_available["benchmark_source_date"]
    + pd.Timedelta(days=1)
)

df_dvol_available = (
    df_dvol_available[
        [
            "benchmark_source_date",
            "benchmark_available_at",
            "dvol",
            "sigma",
        ]
    ]
    .sort_values("benchmark_available_at")
    .drop_duplicates(
        subset=["benchmark_available_at"],
        keep="last",
    )
)


# ------------------------------------------------------------
# Match latest DVOL close available before each Kalshi trade
# ------------------------------------------------------------

df_kalshi_intraday = pd.merge_asof(
    df_kalshi_intraday.sort_values(
        "observation_time"
    ),
    df_dvol_available,
    left_on="observation_time",
    right_on="benchmark_available_at",
    direction="backward",
)

df_kalshi_intraday["dvol_age_hours"] = (
    (
        df_kalshi_intraday["observation_time"]
        - df_kalshi_intraday["benchmark_available_at"]
    ).dt.total_seconds()
    / 3600
)


# ------------------------------------------------------------
# Select and validate benchmark output
# ------------------------------------------------------------

kalshi_intraday_benchmark_cols = [
    "ticker",
    "event_ticker",
    "observation_time",
    "spot_available_at",
    "spot_age_seconds",
    "spot",
    "benchmark_source_date",
    "benchmark_available_at",
    "dvol_age_hours",
    "dvol",
    "sigma",
]

df_kalshi_intraday_benchmark = (
    df_kalshi_intraday[
        kalshi_intraday_benchmark_cols
    ]
    .sort_values(["observation_time", "ticker"])
    .reset_index(drop=True)
)

if df_kalshi_intraday_benchmark[
    ["spot", "dvol", "sigma"]
].isna().any().any():
    raise ValueError(
        "Missing Kalshi intraday benchmark inputs:\n"
        + str(
            df_kalshi_intraday_benchmark[
                ["spot", "dvol", "sigma"]
            ].isna().sum()
        )
    )

if not (
    df_kalshi_intraday_benchmark["spot_available_at"]
    <= df_kalshi_intraday_benchmark["observation_time"]
).all():
    raise ValueError("Spot look-ahead detected.")

if not (
    df_kalshi_intraday_benchmark["benchmark_available_at"]
    <= df_kalshi_intraday_benchmark["observation_time"]
).all():
    raise ValueError("DVOL look-ahead detected.")

assert len(df_kalshi_intraday_benchmark) == 3_135
assert df_kalshi_intraday_benchmark["ticker"].is_unique
assert (
    df_kalshi_intraday_benchmark["spot_age_seconds"] >= 0
).all()

kalshi_intraday_benchmark_path = (
    FINAL_DIR / "deribit_kalshi_intraday_inputs.csv"
)

df_kalshi_intraday_benchmark.to_csv(
    kalshi_intraday_benchmark_path,
    index=False,
)

print("=" * 70)
print("Kalshi temporally available intraday benchmark")
print("=" * 70)

print(f"Rows: {len(df_kalshi_intraday_benchmark):,}")
print(
    f"Events: "
    f"{df_kalshi_intraday_benchmark['event_ticker'].nunique():,}"
)

print("\nSpot age in seconds:")
display(
    df_kalshi_intraday_benchmark[
        "spot_age_seconds"
    ].describe()
)

print("\nDVOL age in hours:")
display(
    df_kalshi_intraday_benchmark[
        "dvol_age_hours"
    ].describe()
)

print("\nMissing inputs:")
print(
    df_kalshi_intraday_benchmark[
        ["spot", "dvol", "sigma"]
    ].isna().sum()
)

print("\nSaved Kalshi intraday benchmark to:")
print(kalshi_intraday_benchmark_path)

Kalshi temporally available intraday benchmark
Rows: 3,135
Events: 485

Spot age in seconds:


count    3135.000000
mean       29.120158
std        17.919626
min         0.010836
25%        12.715739
50%        29.570288
75%        44.614176
max        59.983175
Name: spot_age_seconds, dtype: float64


DVOL age in hours:


count    3135.000000
mean        3.456292
std         0.480972
min         3.000603
25%         3.053304
50%         3.194515
75%         4.029879
max         4.499335
Name: dvol_age_hours, dtype: float64


Missing inputs:
spot     0
dvol     0
sigma    0
dtype: int64

Saved Kalshi intraday benchmark to:
/Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/deribit_kalshi_intraday_inputs.csv


In [32]:
# ============================================================
# Final Deribit export summary
# ============================================================

required_output_paths = [
    btc_spot_raw_path,
    btc_dvol_raw_path,
    kalshi_spot_1m_raw_path,
    deribit_daily_final_path,
    kalshi_intraday_benchmark_path,
]

missing_output_paths = [
    path for path in required_output_paths
    if not path.exists()
]

if missing_output_paths:
    raise FileNotFoundError(
        "Missing Deribit output files:\n"
        + "\n".join(map(str, missing_output_paths))
    )


# ------------------------------------------------------------
# Final quality checks
# ------------------------------------------------------------

daily_missing_inputs = (
    df_deribit_daily[
        ["spot", "dvol", "sigma"]
    ]
    .isna()
    .sum()
    .sum()
)

daily_duplicate_dates = (
    df_deribit_daily["date"]
    .duplicated()
    .sum()
)

intraday_missing_inputs = (
    df_kalshi_intraday_benchmark[
        ["spot", "dvol", "sigma"]
    ]
    .isna()
    .sum()
    .sum()
)

intraday_duplicate_tickers = (
    df_kalshi_intraday_benchmark["ticker"]
    .duplicated()
    .sum()
)

invalid_intraday_spot = (
    df_kalshi_intraday_benchmark["spot"] <= 0
).sum()

invalid_intraday_sigma = (
    df_kalshi_intraday_benchmark["sigma"] <= 0
).sum()

spot_lookahead_violations = (
    df_kalshi_intraday_benchmark["spot_available_at"]
    > df_kalshi_intraday_benchmark["observation_time"]
).sum()

dvol_lookahead_violations = (
    df_kalshi_intraday_benchmark["benchmark_available_at"]
    > df_kalshi_intraday_benchmark["observation_time"]
).sum()

negative_spot_age = (
    df_kalshi_intraday_benchmark["spot_age_seconds"] < 0
).sum()

negative_dvol_age = (
    df_kalshi_intraday_benchmark["dvol_age_hours"] < 0
).sum()


# Stop before printing successful completion if anything is wrong.
assert len(df_deribit_daily) == 886
assert daily_missing_inputs == 0
assert daily_duplicate_dates == 0

assert len(df_kalshi_intraday_benchmark) == 3_135
assert (
    df_kalshi_intraday_benchmark["event_ticker"].nunique()
    == 485
)
assert intraday_duplicate_tickers == 0
assert intraday_missing_inputs == 0
assert invalid_intraday_spot == 0
assert invalid_intraday_sigma == 0
assert spot_lookahead_violations == 0
assert dvol_lookahead_violations == 0
assert negative_spot_age == 0
assert negative_dvol_age == 0


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("=" * 70)
print("Final Deribit export summary")
print("=" * 70)

print("Daily raw files:")
print(f"  BTC spot daily: {btc_spot_raw_path}")
print(f"  BTC DVOL daily: {btc_dvol_raw_path}")

print("\nIntraday raw file:")
print(f"  Kalshi BTC spot 1m: {kalshi_spot_1m_raw_path}")

print("\nFinal files:")
print(f"  Daily inputs: {deribit_daily_final_path}")
print(
    f"  Kalshi intraday benchmark: "
    f"{kalshi_intraday_benchmark_path}"
)

print("\nDaily sample:")
print(f"  Rows: {len(df_deribit_daily):,}")
print(f"  Unique dates: {df_deribit_daily['date'].nunique():,}")
print(
    f"  Date range: "
    f"{df_deribit_daily['date'].min()} -> "
    f"{df_deribit_daily['date'].max()}"
)
print(f"  Missing inputs: {daily_missing_inputs:,}")
print(f"  Duplicate dates: {daily_duplicate_dates:,}")

print("\nKalshi intraday sample:")
print(f"  Rows: {len(df_kalshi_intraday_benchmark):,}")
print(
    f"  Events: "
    f"{df_kalshi_intraday_benchmark['event_ticker'].nunique():,}"
)
print(
    f"  Tickers: "
    f"{df_kalshi_intraday_benchmark['ticker'].nunique():,}"
)
print(f"  Missing inputs: {intraday_missing_inputs:,}")
print(f"  Duplicate tickers: {intraday_duplicate_tickers:,}")
print(f"  Invalid spot: {invalid_intraday_spot:,}")
print(f"  Invalid sigma: {invalid_intraday_sigma:,}")
print(
    f"  Spot look-ahead violations: "
    f"{spot_lookahead_violations:,}"
)
print(
    f"  DVOL look-ahead violations: "
    f"{dvol_lookahead_violations:,}"
)

print("\nInput-age summary:")
print(
    "  Maximum spot age in seconds:",
    round(
        df_kalshi_intraday_benchmark[
            "spot_age_seconds"
        ].max(),
        3,
    ),
)
print(
    "  Maximum DVOL age in hours:",
    round(
        df_kalshi_intraday_benchmark[
            "dvol_age_hours"
        ].max(),
        3,
    ),
)

print("\n03_deribit_data.ipynb completed successfully.")

Final Deribit export summary
Daily raw files:
  BTC spot daily: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/btc_spot_daily.csv
  BTC DVOL daily: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/btc_dvol_daily.csv

Intraday raw file:
  Kalshi BTC spot 1m: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw/binance_btcusdt_1m_kalshi_windows.csv

Final files:
  Daily inputs: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/deribit_btc_daily_inputs.csv
  Kalshi intraday benchmark: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/deribit_kalshi_intraday_inputs.csv

Daily sample:
  Rows: 886
  Unique dates: 886
  Date range: 2024-01-01 00:00:00+00:00 -> 2026-06-04 00:00:00+00:00
  Missing inputs: 0
  Duplicate dates: 0

Kalshi intraday sample:
  Rows: 3,135
  Events: 485
  Tickers: 3,135
  Missing inputs: 0
  Duplicate tickers: 0
  Invalid spot: 0
  Invalid sigma: 0
  Spot look-ahead violations: 0
  DVOL lo